# Clase 4 · Las calles de Bogotá (gemelo)

**Taller en Ciencia de Datos y Aprendizaje Automático** · Especialización en Estadística · UNAL Manizales · 2026-II
Profesor: Carlos Manuel Orrego Franco

Este cuaderno lo resuelve cada uno, de forma individual, en la segunda mitad de la clase. Es el gemelo de *El Mundial de Qatar*: la misma técnica en otro mundo.

| Ficha | |
|---|---|
| **Objetivo pedagógico** | Repetir por cuenta propia el camino de la clase con un evento más raro y más serio que un gol: modelar la probabilidad de que un siniestro vial sea mortal, medirla con reglas de puntuación, revisar su calibración y usar la suma de probabilidades para decidir dónde intervenir. |
| **Enfoque** | Machine Learning. |
| **Resultados de aprendizaje** | 1 (formular el problema, los objetivos y la métrica), 2 (seleccionar y preparar datos), 4 (implementar y evaluar un modelo), 5 (comunicar a quien no es técnico) y 6 (reflexión ética, técnica y operativa). |
| **Uso de IA** | Pueden pedirle a Claude que explique una celda, que les ayude a llenar un hueco o que interprete un error. Deben poder explicar sin ayuda por qué se quitan los siniestros de solo daños, por qué la exactitud no sirve aquí, qué le hace el balanceo a la probabilidad, cómo se lee la curva de calibración y con qué criterio escogieron los cruces. |
| **Entregable** | El cuaderno completo y corrido en su portafolio de GitHub, con las predicciones escritas, la decisión en tres frases, la bitácora y la tabla de la sección 4.4 llenas. |

**Cómo se trabaja.** El código está escrito casi todo. Donde vean `____` falta una pieza, y el comentario de al lado dice qué va. Lean la celda entera antes de llenar. Si una celda falla, lean el error antes de pedir ayuda.

> **Antes de empezar: conozcan la fuente.** Los siniestros de hoy salen de la base del Anuario de Siniestralidad Vial de 2024. La Secretaría Distrital de Movilidad la publica en [la página de anuarios de su Observatorio](https://observatorio.movilidadbogota.gov.co/publicaciones/anuarios), junto al informe de cada año. La base es un archivo de Excel que trae su propio diccionario en una hoja aparte. Abran la página y explórenla cinco minutos antes de correr nada. Respondan mirando la fuente:
>
> - ¿Qué entidad publica los anuarios y desde qué año hay uno disponible?
> - ¿Qué dos archivos acompañan a cada año, y cuál de los dos es el dato?
> - ¿Cada cuánto sale un anuario nuevo, y cuál es el más reciente?

In [ ]:
# Celda de arranque. No hay que cambiarle nada: trae las herramientas del curso.
import pathlib, subprocess, sys
raiz = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, pathlib.Path.cwd() / "taller-cd-ml-2026-2"] if (p / "taller" / "datos.py").exists()), None)
if raiz is None:
    subprocess.run(["git", "clone", "-q", "https://github.com/cmorregof/taller-cd-ml-2026-2.git"], check=True)
    raiz = pathlib.Path("taller-cd-ml-2026-2")
sys.path.insert(0, str(raiz))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from taller import descargar, procedencia

PALETA = ["#1D4E89", "#D7263D", "#1B998B", "#6C4AB6", "#5BC0EB", "#F49097", "#9AA5B1"]      # colores del curso
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": plt.cycler(color=PALETA), "axes.titleweight": "bold", "figure.dpi": 110})
print("Herramientas listas.")

## 1. Formulación

La Secretaría Distrital de Movilidad de Bogotá publica cada siniestro vial que registra la policía de tránsito: dónde fue, a qué hora, qué clase de siniestro fue, quiénes estaban involucrados y qué tan grave resultó. Detrás de cada fila hay personas, y detrás de algunas, una muerte. Conviene trabajar este dato con ese cuidado.

Imagínense el equipo de seguridad vial de la Secretaría. Tiene recursos para poner controles de velocidad y cruces seguros en cien puntos de la ciudad. Necesita tres respuestas:

1. **Predecir.** Cuando ocurre un siniestro con víctimas, ¿qué probabilidad hay de que sea mortal?
2. **Explicar.** ¿Qué condiciones acompañan a los siniestros mortales?
3. **Decidir.** ¿En qué cien cruces se ponen primero los controles?

**La respuesta.** $y_i=1$ si el siniestro $i$ dejó al menos una persona muerta, contando las muertes de los treinta días siguientes. El modelo entrega $\hat p_i$.

**La métrica.** Como en clase, lo que importa es la probabilidad, porque se va a sumar. El número esperado de siniestros mortales en un cruce $c$ es

$$\hat m_c=\sum_{i\in c}\hat p_i,$$

y esa suma solo vale si las probabilidades están calibradas. Medimos con el puntaje de Brier y con la pérdida logarítmica:

$$\text{Brier}=\frac{1}{n}\sum_i(\hat p_i-y_i)^2,\qquad \text{pérdida logarítmica}=-\frac{1}{n}\sum_i\bigl[y_i\log\hat p_i+(1-y_i)\log(1-\hat p_i)\bigr].$$

### Predicción escrita

Escriban su apuesta antes de correr nada.

- De cada cien siniestros con víctimas en Bogotá, ¿cuántos son mortales? **Mi apuesta:**
- ¿Qué es más letal: un siniestro con peatón, con motocicleta o con vehículo de carga? **Mi apuesta:**
- Cien cruces son una fracción mínima de la ciudad. ¿Qué porcentaje de los siniestros mortales futuros ocurrirá en los cien cruces que ustedes escojan? **Mi apuesta:**

## 2. Datos

Una línea trae la tabla completa, con diez años de siniestros. **La primera vez tarda cerca de un minuto y medio.** Córranla y lean la sección mientras termina.

In [ ]:
ACTOR = "todos"              # la población del análisis. Al final lo cambiarán por "Con_Peaton"
CONTROLES = 100              # cuántos cruces alcanza a intervenir la Secretaría

In [ ]:
crudo = descargar("siniestros_bogota")
print("Filas:", len(crudo), "| Columnas:", crudo.shape[1], "| ¿Algún siniestro repetido?", crudo["Codigo_Accidente"].duplicated().any())
crudo.head(3).T

### 2.1 Las trampas del dato

**Primera trampa: una fecha que es un número.** Miren `Fecha_Acc`. Excel guarda las fechas como el número de días transcurridos desde el 30 de diciembre de 1899, y al leer el archivo llega ese número.

In [ ]:
crudo["fecha"] = pd.to_datetime(____) + pd.to_timedelta(crudo["Fecha_Acc"], unit="D")      # HUECO 1: el día cero de Excel, entre comillas, como año-mes-día
print("Primer siniestro:", crudo["fecha"].min().date(), "| último:", crudo["fecha"].max().date())
print("Filas en que el año de la fecha no coincide con la columna AA_Acc:", int((crudo["fecha"].dt.year != crudo["AA_Acc"]).sum()))

**Compruébenlo ustedes mismos.** Las fechas deben ir de 2015 a 2024 y el año debe coincidir con `AA_Acc` en todas las filas. Si no, revisen el hueco 1.

**Segunda trampa: una categoría que desaparece.** La gravedad tiene tres valores. Miren cuántos siniestros hay de cada uno por año.

In [ ]:
por_anio = pd.crosstab(crudo["AA_Acc"], crudo["Gravedad_indicador_30d"])
display(por_anio)
por_anio.plot(figsize=(9, 3.4), color=["#1B998B", "#D7263D", "#1D4E89"], marker="o"); plt.title("Siniestros registrados por año, según la gravedad"); plt.xlabel(""); plt.show()

Los siniestros de solo daños se desploman en 2023. La ciudad no se volvió más prudente de un año para otro. Lo que cambió fue la norma sobre el registro de los choques sin víctimas, y desde entonces casi no entran a esta tabla. Una serie que mezcle los tres valores mostraría una caída de la siniestralidad que nunca ocurrió.

**Decisión:** el análisis usa solo los siniestros **con víctimas**, que se registran igual durante los diez años.

**Tercera trampa: dos maneras de contar a los muertos.** Hay dos columnas de gravedad.

In [ ]:
print(pd.crosstab(crudo["Gravedad_Indicador_Tradicional"], crudo["Gravedad_indicador_30d"]).to_string())

El diccionario del anuario dice que el segundo indicador mide la gravedad «tomando en cuenta los fallecimientos que ocurren dentro de los 30 días posteriores al siniestro». Las dos columnas discrepan en unos pocos cientos de filas. Miren en qué dirección: hay siniestros que el indicador tradicional marca con muertos y el de treinta días marca solo con heridos. El diccionario no explica esas filas, y yo no voy a inventar la explicación. **Decisión:** se usa el indicador de treinta días, que es el que el anuario define con precisión, y la duda queda anotada en los riesgos.

**Cuarta trampa: un vacío que significa «no».** Las columnas que empiezan por `Con_` dicen si en el siniestro había una moto, un peatón, un vehículo de carga.

In [ ]:
banderas = [c for c in crudo.columns if c.startswith("Con_")]
print(pd.DataFrame({"valores distintos": [crudo[c].dropna().unique().tolist() for c in banderas],
                    "% de filas con SI": [(crudo[c] == "SI").mean() * 100 for c in banderas]}, index=banderas).round(1).to_string())

Solo aparece el valor `SI`. Donde no hubo moto la celda está vacía. En la clase 2 un vacío era un faltante. Aquí el vacío es la respuesta «no», y tratarlo como faltante borraría casi todo el dato. La limpieza depende de saber cómo se llenó la tabla.

Tres de esas columnas son distintas de las demás. `Con_Embriaguez`, `Con_Velocidad` y `Con_Huecos` no describen quién estaba en el siniestro. Registran, según el diccionario del anuario, que hubo una **hipótesis** sobre la causa. Una hipótesis se formula cuando ya se conoce el resultado del siniestro. **Decisión:** no entran al modelo. La sección 3.5 les pide medir cuánto cambiarían el resultado.

**Quinta trampa: un punto del mapa que no es un cruce.** Cada siniestro trae latitud y longitud. Miren cuáles pares de coordenadas se repiten más, idénticos hasta el último decimal, y de cuántas localidades distintas vienen sus siniestros.

In [ ]:
pares = (crudo.groupby(["Latitud", "Longitud"]).agg(siniestros=("Localidad", "size"), localidades=("Localidad", "nunique"), direcciones=("Direccion", "nunique"))
         .sort_values("siniestros", ascending=False))
display(pares.head(5))
sospechosos = pares[pares["localidades"] >= 5].index
en_sospechoso = pd.MultiIndex.from_frame(crudo[["Latitud", "Longitud"]]).isin(sospechosos)
print(f"Pares de coordenadas con siniestros de cinco o más localidades: {len(sospechosos)} | siniestros afectados: {en_sospechoso.sum()}")
print("Años de esos siniestros:", crudo.loc[en_sospechoso, "AA_Acc"].value_counts().sort_index().to_dict())
crudo.loc[en_sospechoso, ["Latitud", "Longitud"]] = np.nan

Un mismo punto no puede quedar en muchas localidades a la vez, ni tener cientos de direcciones distintas. Ese par de coordenadas es un valor por defecto: lo que el sistema escribió cuando no pudo ubicar la dirección. Es otro faltante disfrazado, como el `-89` de la clase 2. Si se deja, aparece como el cruce más peligroso de Bogotá, y la Secretaría mandaría el primer control a un lugar que no existe.

**Decisión:** las coordenadas que reúnen siniestros de cinco o más localidades se declaran faltantes. El siniestro se conserva para el modelo y queda por fuera del mapa.

### 2.2 La tabla limpia

In [ ]:
victimas = crudo[crudo["Gravedad_indicador_30d"] != ____].copy()                 # HUECO 2: la categoría que se deja por fuera, como aparece en la tabla
if ACTOR != "todos":
    victimas = victimas[victimas[ACTOR] == "SI"]
victimas = victimas.sort_values("fecha").reset_index(drop=True)
y = (victimas[____] == "Con Muertos").astype(int)                   # HUECO 3: la columna de gravedad que cuenta las muertes de los 30 días siguientes

hora = victimas["Hora_Acc"]
X = pd.DataFrame({
    "noche": ((hora >= 21) | (hora < 5)).astype(int),
    "hora_sen": np.sin(2 * np.pi * hora / 24), "hora_cos": np.cos(2 * np.pi * hora / 24),
    "fin_de_semana": victimas["Dia_Semana_Acc"].isin(["sábado", "domingo"]).astype(int),
    "atropello": (victimas["Clase_Acc"] == "Atropello").astype(int),
    "choque": (victimas["Clase_Acc"] == "Choque").astype(int),
    "volcamiento": (victimas["Clase_Acc"] == "Volcamiento").astype(int),
    "contra_objeto_fijo": (victimas["Elemento_Choque"] == "Objeto Fijo").astype(int),
})
actores = {"peaton": "Con_Peaton", "moto": "Con_Moto", "bicicleta": "Con_Bicicleta", "carga": "Con_Carga", "bus": "Con_Tpp",
           "taxi": "Con_Tpi", "persona_mayor": "Con_Persona_Mayor", "menor": "Con_Menores"}
for nombre, columna in actores.items():
    X[nombre] = (victimas[columna] == ____).astype(int)                                 # HUECO 4: el único valor que aparece en las columnas Con_
hipotesis = pd.DataFrame({nombre: (victimas[columna] == "SI").astype(int) for nombre, columna in
                          {"embriaguez": "Con_Embriaguez", "velocidad": "Con_Velocidad", "huecos": "Con_Huecos"}.items()})
variables = list(X.columns)

print(f"Población: {ACTOR} | siniestros con víctimas: {len(victimas)} | mortales: {y.sum()} | tasa: {y.mean() * 100:.2f} %")
print(f"Sin coordenadas: {victimas['Latitud'].isna().mean() * 100:.1f} %")

Comparen con su primera apuesta. En el Mundial, uno de cada nueve remates era gol. Aquí el evento es bastante más raro, y todo lo que vimos en clase sobre el desbalance pega más fuerte.

### 2.3 Conocer el dato antes de modelarlo

Cuatro miradas: cuántos siniestros hay y qué tan letales son según el año, la hora y quién estaba involucrado, y dónde ocurren. El error estándar de una tasa $\hat p$ con $n$ casos es $\sqrt{\hat p(1-\hat p)/n}$.

In [ ]:
def tasa_por(grupo):
    tabla = pd.DataFrame({"n": y.groupby(grupo).size(), "tasa": y.groupby(grupo).____() * 100})      # HUECO 5: la operación que convierte ceros y unos en una fracción
    tabla["ee"] = np.sqrt(tabla["tasa"] * (100 - tabla["tasa"]) / tabla["n"])
    return tabla

fig, ejes = plt.subplots(2, 2, figsize=(13, 8))
anual = tasa_por(victimas["AA_Acc"])
ejes[0, 0].bar(anual.index, anual["n"], color="#1D4E89"); ejes[0, 0].set(title="Siniestros con víctimas por año", ylabel="siniestros")
otro = ejes[0, 0].twinx(); otro.errorbar(anual.index, anual["tasa"], yerr=2 * anual["ee"], color="#D7263D", marker="o"); otro.set_ylabel("% mortal", color="#D7263D"); otro.spines["right"].set_visible(True)

horaria = tasa_por(hora)
ejes[0, 1].bar(horaria.index, horaria["n"], color="#1D4E89"); ejes[0, 1].set(title="Por hora del día", xlabel="hora", ylabel="siniestros")
otro = ejes[0, 1].twinx(); otro.errorbar(horaria.index, horaria["tasa"], yerr=2 * horaria["ee"], color="#D7263D", marker="o"); otro.set_ylabel("% mortal", color="#D7263D"); otro.spines["right"].set_visible(True)

todas = pd.concat([X.drop(columns=["hora_sen", "hora_cos"]), hipotesis], axis=1)
letalidad = pd.DataFrame({"n": todas.sum(), "tasa": [y[todas[c] == 1].mean() * 100 if todas[c].sum() else np.nan for c in todas]}).dropna().sort_values("tasa")
letalidad["ee"] = np.sqrt(letalidad["tasa"] * (100 - letalidad["tasa"]) / letalidad["n"])
ejes[1, 0].barh([f"{nombre} (n={int(n)})" for nombre, n in zip(letalidad.index, letalidad["n"])], letalidad["tasa"], xerr=2 * letalidad["ee"],
                color=["#D7263D" if nombre in hipotesis else "#1B998B" for nombre in letalidad.index], ecolor="#1D4E89")
ejes[1, 0].axvline(y.mean() * 100, color="gray", ls=":"); ejes[1, 0].set(title="% mortal cuando la condición está presente (rojo: hipótesis)", xlabel="%"); ejes[1, 0].tick_params(axis="y", labelsize=8)

con_lugar = victimas["Latitud"].notna()
ejes[1, 1].hexbin(victimas.loc[con_lugar, "Longitud"], victimas.loc[con_lugar, "Latitud"], gridsize=70, cmap="Blues", mincnt=1, bins="log")
ejes[1, 1].scatter(victimas.loc[con_lugar & (y == 1), "Longitud"], victimas.loc[con_lugar & (y == 1), "Latitud"], s=2, color="#1D4E89", alpha=0.4)
ejes[1, 1].set(title="Dónde ocurren (puntos oscuros: mortales)", xlim=(-74.23, -74.0), ylim=(4.45, 4.84)); ejes[1, 1].set_xticks([]); ejes[1, 1].set_yticks([])
plt.tight_layout(); plt.show()

Comparen con su segunda apuesta y respondan en una línea cada una.

- ¿A qué hora ocurren más siniestros y a qué hora son más letales? ¿Coinciden? **Mi lectura:**
- ¿Qué condición tiene la tasa de mortalidad más alta? ¿Es una condición del siniestro o una hipótesis del agente? **Mi lectura:**
- ¿La tasa de mortalidad cambia entre años más de lo que explican sus barras de error? **Mi lectura:**

**¿Quiénes están en la tabla y quiénes no?** Un siniestro entra al registro si llega la policía de tránsito. Respondan: ¿qué clase de siniestro con heridos puede quedar sin registrar? ¿Eso infla o desinfla la tasa de mortalidad que acaban de calcular? **Mi respuesta:**

### 2.4 Agrupar sin etiquetas: ¿cuántas clases de siniestro hay?

Como en clase, $K$-medias agrupa sin mirar la respuesta. Busca $K$ centros que minimicen

$$\sum_{i=1}^{n}\min_k\lVert z_i-\mu_k\rVert^2,$$

con $z_i$ las variables estandarizadas. Después miramos qué tan letal es cada grupo.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

describen = ["noche", "atropello", "choque", "peaton", "moto", "bicicleta", "carga", "bus", "persona_mayor"]
grupo = pd.Series(KMeans(n_clusters=____, n_init=5, random_state=0).fit_predict(StandardScaler().fit_transform(X[describen])), index=X.index)   # HUECO 6: cuántos grupos pedir. Usen cinco
perfil = X[describen].groupby(grupo).mean().round(2)
perfil.insert(0, "% mortal", (y.groupby(grupo).mean() * 100).round(2))
perfil.insert(0, "siniestros", grupo.value_counts())
display(perfil.sort_values("% mortal", ascending=False))

Cada fila es un grupo, y cada columna dice qué fracción de sus siniestros tiene esa condición. Pónganle un nombre a cada grupo en dos o tres palabras, y digan cuál es el más letal y cuál el más numeroso. **Mis nombres:**

## 3. Modelado

### 3.1 La partición y la trampa de la exactitud

Aquí manda el tiempo. La Secretaría va a usar el modelo con siniestros posteriores a los del entrenamiento:

$$\underbrace{2015\ \dots\ 2020}_{\text{entrenamiento}}\ \Big|\ \underbrace{2021\ \dots\ 2022}_{\text{validación: se escoge}}\ \Big|\ \underbrace{2023\ \dots\ 2024}_{\text{prueba: se reporta}}$$

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss

anio = victimas["AA_Acc"].values
E, V, P = anio <= 2020, (anio >= 2021) & (anio <= ____), anio >= 2023                 # HUECO 7: el último año de la validación
for nombre, tramo in [("Entrenamiento", E), ("Validación", V), ("Prueba", P)]:
    print(f"{nombre:14s} {tramo.sum():6d} siniestros | {y[tramo].sum():5d} mortales | tasa {y[tramo].mean() * 100:.2f} %")

logistica = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X[E], y[E])
prob = logistica.predict_proba(X[V])[:, 1]
real = y[V].values
print(f"\nExactitud de decir que ningún siniestro es mortal: {(real == 0).mean():.3f}")
print(f"Exactitud de la regresión logística con umbral 0,5: {((prob >= 0.5) == real).mean():.3f}")
print(f"Siniestros a los que el modelo les da más de 50 %: {(prob >= 0.5).sum()} | probabilidad más alta que entrega: {prob.max():.2f}")

Respondan: ¿por qué la exactitud de «ninguno es mortal» es tan alta? ¿Qué le diría esa cifra a un periodista que no conoce el dato? **Mi respuesta:**

### 3.2 La línea base

Dos referencias sin aprendizaje automático. La primera le da a todo siniestro la tasa de mortalidad del entrenamiento. La segunda le da la tasa de su clase de siniestro (atropello, choque, volcamiento y demás), calculada en el entrenamiento.

In [ ]:
def puntuar(real, prob):
    prob = np.clip(np.asarray(prob, dtype=float), 1e-6, 1 - 1e-6)
    return {"pérdida logarítmica": log_loss(real, prob), "Brier": brier_score_loss(real, prob),
            "AUC": roc_auc_score(real, prob) if len(np.unique(prob)) > 1 else 0.5, "mortales que predice": prob.____(), "mortales que hubo": int(np.sum(real))}    # HUECO 8: la operación que convierte probabilidades en un número esperado de casos

clase = victimas["Clase_Acc"]
def tasa_de_su_clase(tramo_entrena, tramo_destino):
    tasas = y[tramo_entrena].groupby(clase[tramo_entrena]).mean()
    return clase[tramo_destino].map(tasas).fillna(y[tramo_entrena].mean()).values

referencias_validacion = {"tasa base": np.full(V.sum(), y[E].mean()), "tasa de su clase": tasa_de_su_clase(E, V)}
probabilidades = {"tasa base": np.full(P.sum(), y[E | V].mean()), "tasa de su clase": tasa_de_su_clase(E | V, P)}
errores = {nombre: {"validación": puntuar(y[V], referencias_validacion[nombre])["pérdida logarítmica"], **puntuar(y[P], probabilidades[nombre])} for nombre in probabilidades}
display(pd.DataFrame(errores).T.round(4))

### 3.3 El catálogo de modelos y el mismo protocolo de la clase

La celda siguiente es el catálogo de candidatos. Es un diccionario, `MODELOS`, con una línea por modelo, y no tiene huecos.

- El texto entre comillas es el **nombre** con el que el modelo aparece en las tablas y en las gráficas.
- `make_pipeline(...)` encadena la **preparación** (por ejemplo `StandardScaler()`, que estandariza cada variable) con el **clasificador**, que va al final. Así la preparación se ajusta solo con los siniestros de entrenamiento.
- Para **agregar un modelo** escriban una línea nueva con otro nombre, importen su clase de scikit-learn al comienzo de la celda y corran el cuaderno desde ahí. Las celdas que siguen recorren lo que haya en el diccionario y no hay que tocarlas.

Aquí la métrica es una probabilidad, así que solo sirven clasificadores que tengan `predict_proba`. Como el evento es muy raro, los vecindarios y las hojas son grandes, por la razón que vimos con los remates lejanos.

**Ejercicio.** Agreguen al menos tres modelos más de scikit-learn y vuelvan a correr. No le cambien el nombre a `"logística"`, porque más abajo el cuaderno la usa por ese nombre.

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier

MODELOS = {
    "logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "logística, castigo fuerte": make_pipeline(StandardScaler(), LogisticRegression(C=0.001, max_iter=2000)),
    "100 vecinos": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=100)),
    "300 vecinos": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=300)),
    "árbol": DecisionTreeClassifier(max_depth=6, min_samples_leaf=200, random_state=0),
    "bosque": RandomForestClassifier(n_estimators=200, min_samples_leaf=50, random_state=0, n_jobs=-1),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=200, min_samples_leaf=50, random_state=0, n_jobs=-1),
    "boosting": HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=200, random_state=0),
    "boosting clásico": GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=0),
    "Bayes ingenuo": BernoulliNB(),
    "discriminante lineal": make_pipeline(StandardScaler(), LinearDiscriminantAnalysis()),
    "red pequeña": make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(16,), early_stopping=True, max_iter=100, random_state=0)),
}
print(len(MODELOS), "modelos en el catálogo:", ", ".join(MODELOS))

Gana quien tenga menor pérdida logarítmica en **validación** y se reporta lo que haga en **prueba**. Esta celda tarda cerca de un minuto.

**Predicción escrita:** ¿cuánto bajará la pérdida logarítmica frente a la tasa base: mucho, poco o casi nada? **Mi apuesta:**

In [ ]:
from sklearn.base import clone

def copia(nombre, peso=None):
    """Una copia sin entrenar del modelo `nombre`. Con peso="balanced", su clasificador pesa las dos clases por igual."""
    modelo = clone(MODELOS[nombre])
    if peso is not None:
        (modelo[-1] if hasattr(modelo, "steps") else modelo).set_params(class_weight=peso)
    return modelo

def admite_peso(nombre):
    """¿El clasificador de `nombre` sabe pesar las clases? Los vecinos y la red, por ejemplo, no."""
    clasificador = MODELOS[nombre][-1] if hasattr(MODELOS[nombre], "steps") else MODELOS[nombre]
    return "class_weight" in clasificador.get_params()

for nombre in MODELOS:
    modelo = copia(nombre).fit(X[E], y[E])
    en_validacion = puntuar(y[V], modelo.predict_proba(X[V])[:, 1])["pérdida logarítmica"]
    modelo = copia(nombre).fit(X[E | V], y[E | V])
    probabilidades[nombre] = modelo.predict_proba(X[P])[:, 1]
    errores[nombre] = {"validación": en_validacion, **puntuar(y[P], probabilidades[nombre])}
todos = pd.DataFrame(errores).T

todos[["validación", "pérdida logarítmica"]].rename(columns={"pérdida logarítmica": "prueba"}).sort_values("validación").plot.barh(figsize=(9, 5.5), color=["#1D4E89", "#1B998B"])
plt.xlabel("pérdida logarítmica (menos es mejor)"); plt.title(f"{len(todos)} candidatos: validación para escoger, prueba para reportar"); plt.gca().invert_yaxis(); plt.show()
display(todos.sort_values("validación").round(4))

mejor = todos[____].idxmin()                                                    # HUECO 9: la columna con la que se escoge al ganador
elegido = mejor if todos.loc[mejor, "pérdida logarítmica"] < todos.loc["tasa de su clase", "pérdida logarítmica"] else "tasa de su clase"
print(f"Ganador en validación: {mejor} | en prueba: pérdida logarítmica {todos.loc[mejor, 'pérdida logarítmica']:.4f}, predice {todos.loc[mejor, 'mortales que predice']:.0f} mortales y hubo {int(todos.loc[mejor, 'mortales que hubo'])}")
print(f"Tasa de su clase: {todos.loc['tasa de su clase', 'pérdida logarítmica']:.4f} | tasa base: {todos.loc['tasa base', 'pérdida logarítmica']:.4f} | método con el que seguimos: {elegido}")

El ganador salió de una lista larga. La gráfica siguiente pone cada modelo del catálogo según su puesto en validación y su puesto en prueba. Si el orden de validación fuera una propiedad firme de los modelos, los puntos caerían sobre la diagonal.

In [ ]:
del_catalogo = todos.loc[list(MODELOS), ["validación", "pérdida logarítmica"]].rename(columns={"pérdida logarítmica": "prueba"})
puestos = del_catalogo.rank(method="first").astype(int)                          # el puesto 1 es la menor pérdida logarítmica
primero = del_catalogo["validación"].idxmin()

fig, eje = plt.subplots(figsize=(5, 5), facecolor="#FBFAF7")
eje.set_facecolor("#FBFAF7")
eje.plot([1, len(puestos)], [1, len(puestos)], color="#9AA5B1", ls=":")
eje.scatter(puestos["validación"], puestos["prueba"], s=45, zorder=3, color=["#D7263D" if nombre == primero else "#1D4E89" for nombre in puestos.index])
for nombre, fila in puestos.iterrows():
    eje.annotate(nombre, (fila["validación"], fila["prueba"]), xytext=(6, 4), textcoords="offset points", fontsize=7)
eje.set(title="¿El orden de validación se repite en la prueba?", xlabel="puesto en validación (1 es el mejor)", ylabel="puesto en prueba",
        xticks=range(1, len(puestos) + 1), yticks=range(1, len(puestos) + 1))
plt.show()
print(f"Pérdida logarítmica en validación del primero y del segundo: {del_catalogo['validación'].nsmallest(2).round(4).tolist()}")

- El punto rojo es el mejor modelo del catálogo en validación. Miren cuánto se alejan los puntos de la diagonal y cuánta ventaja le sacó el primero al segundo. Con tantos candidatos, ¿cuánto del ganador es azar? **Mi respuesta:**
- El AUC del ganador puede ser alto y la mejora en pérdida logarítmica pequeña. ¿Cómo pueden ser ciertas las dos cosas con un evento tan raro? **Mi respuesta:**
- ¿El ganador predice bien el **total** de siniestros mortales de la prueba? **Mi lectura:**

### 3.4 La trampa de la clase: balancear rompe la probabilidad

En el Mundial, balancear multiplicó por tres los goles que el modelo predecía. La cuenta de la clase decía que las probabilidades se inflan por un factor cercano a $(1-\pi)/\pi$ en la escala de la razón de probabilidades, donde $\pi$ es la tasa del evento.

**Predicción escrita:** con la tasa de mortalidad que calcularon, ¿por cuánto se multiplicará el total de siniestros mortales que predice el modelo balanceado? **Mi apuesta:**

In [ ]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

nombre_modelo = elegido if elegido in MODELOS else "logística"                # si ganó la línea base, se sigue con el modelo más simple del catálogo
para_balancear = nombre_modelo if admite_peso(nombre_modelo) else "logística"  # si el elegido no sabe pesar las clases, la trampa se muestra con la logística
normal = copia(para_balancear).fit(X[E], y[E])
balanceado = copia(para_balancear, peso=____).fit(X[E], y[E])                      # HUECO 10: el valor de class_weight que iguala el peso de las dos clases, entre comillas
p_normal, p_balanceado = normal.predict_proba(X[P])[:, 1], balanceado.predict_proba(X[P])[:, 1]

platt = LogisticRegression().fit(logit(balanceado.predict_proba(X[V])[:, 1]).reshape(-1, 1), y[V])     # recalibración ajustada en validación
p_recalibrado = platt.predict_proba(logit(p_balanceado).reshape(-1, 1))[:, 1]

trampa = pd.DataFrame({"sin balancear": puntuar(y[P], p_normal), "balanceado": puntuar(y[P], p_balanceado),
                       "balanceado y recalibrado": puntuar(y[P], p_recalibrado)}).T
print("Modelo:", para_balancear, "| factor que anticipa la cuenta de la clase:", round((1 - y[E].mean()) / y[E].mean(), 1))
display(trampa.round(4))

- ¿Qué columna casi no cambia entre las tres filas y cuál cambia muchísimo? **Mi lectura:**
- Si la Secretaría le reportara al Concejo el total de la segunda fila como «siniestros mortales esperados», ¿qué pasaría? **Mi respuesta:**

### 3.5 Calibración y la pregunta por las hipótesis

La curva de calibración agrupa los siniestros de la prueba en tramos según su probabilidad y compara la probabilidad promedio $\bar p_b$ con la fracción real de mortales $\bar y_b$. Su resumen es

$$\text{ECE}=\sum_b\frac{n_b}{n}\,\bigl|\bar y_b-\bar p_b\bigr|.$$

In [ ]:
def calibracion(real, prob, tramos=10):
    cortes = pd.qcut(pd.Series(prob), tramos, duplicates="drop")
    tabla = pd.DataFrame({"prob": np.asarray(prob), "real": np.asarray(real)}).groupby(cortes.values, observed=True).agg(n=("real", "size"), predicha=("prob", "mean"), observada=("real", "mean"))
    return tabla, float((tabla["n"] / tabla["n"].sum() * (tabla["observada"] - tabla["____"]).abs()).sum())      # HUECO 11: la columna que se compara con la fracción observada

fig, (izq, der) = plt.subplots(1, 2, figsize=(12, 4.6))
for eje, limite in [(izq, 1.0), (der, 0.25)]:
    for (nombre, prob), color in zip({"sin balancear": p_normal, "balanceado": p_balanceado, "balanceado y recalibrado": p_recalibrado}.items(), ["#1D4E89", "#D7263D", "#5BC0EB"]):
        tabla, ece = calibracion(y[P], prob)
        eje.plot(tabla["predicha"], tabla["observada"], marker="o", color=color, label=f"{nombre} (ECE {ece:.3f})")
    eje.plot([0, 1], [0, 1], color="gray", ls=":"); eje.set(xlim=(0, limite), ylim=(0, 0.25), xlabel="probabilidad promedio que dio el modelo", ylabel="fracción de mortales que hubo")
izq.set_title("Curva de calibración en la prueba"); der.set_title("La misma, ampliada"); izq.legend(fontsize=8); plt.tight_layout(); plt.show()

con_hipotesis = pd.concat([X, hipotesis], axis=1)
modelo_h = copia(nombre_modelo).fit(con_hipotesis[E | V], y[E | V])
comparacion = pd.DataFrame({"sin hipótesis": puntuar(y[P], probabilidades[nombre_modelo]), "con hipótesis": puntuar(y[P], modelo_h.predict_proba(con_hipotesis[P])[:, 1])}).T
display(comparacion.round(4))

- ¿Cuál de las tres curvas promete más de lo que cumple? **Mi lectura:**
- La tabla compara el modelo con y sin las hipótesis del agente. ¿Cuánto mejoran? La hipótesis de exceso de velocidad se formula cuando ya se sabe cómo terminó el siniestro, y es razonable pensar que un siniestro con un muerto se investiga más a fondo. ¿Qué nombre tiene ese problema desde la clase 2, y por qué dejamos esas columnas por fuera aunque ayuden? **Mi respuesta:**

### 3.6 Explicar: ¿qué acompaña a un siniestro mortal?

Con las variables estandarizadas, $\text{OR}_j=e^{\beta_j}$ es la razón de probabilidades por cada desviación estándar de la variable $j$, con las demás quietas.

In [ ]:
explicativo = copia("logística").fit(X[E | V], y[E | V])
razones = pd.Series(np.____(explicativo[-1].coef_[0]), index=variables).sort_values()        # HUECO 12: la función que convierte un coeficiente en una razón de probabilidades
razones.plot.barh(figsize=(8, 5.5), color=["#5BC0EB" if r < 1 else "#D7263D" for r in razones]); plt.axvline(1, color="gray")
plt.xlabel("razón de probabilidades por desviación estándar"); plt.title("Qué acompaña a un siniestro mortal"); plt.show()
print(razones.round(2).to_string())

- ¿Qué tres condiciones suben más la probabilidad de que el siniestro sea mortal? **Mi lectura:**
- «Peatón» y «atropello» casi siempre van juntos. ¿Qué le hace eso a la lectura de cada coeficiente por separado? **Mi respuesta:**
- Que los siniestros con menores sean menos letales, ¿quiere decir que los niños corren menos peligro en la calle? Propongan otra explicación. **Mi respuesta:**

## 4. Comunicación

### 4.1 Decidir: ¿en qué cien cruces?

Llamo **cruce** a una celda de unos 110 metros de lado, que sale de redondear la latitud y la longitud a tres decimales. Con los siniestros de 2015 a 2022 ordeno los cruces de tres maneras y me quedo con los primeros:

1. Por número de siniestros con víctimas.
2. Por número de siniestros mortales.
3. Por número **esperado** de siniestros mortales, $\hat m_c=\sum_{i\in c}\hat p_i$, con las probabilidades del modelo.

Después miro cuántos siniestros mortales de 2023 y 2024 ocurrieron en los cruces escogidos por cada criterio. Ningún criterio conoce esos dos años. La tercera manera es la que necesita probabilidades calibradas: una suma de probabilidades infladas no es un número de muertes.

In [ ]:
modelo_final = copia(nombre_modelo).fit(X[E | V], y[E | V])
victimas["p"] = modelo_final.predict_proba(X)[:, 1]
victimas["mortal"] = y
victimas["cruce"] = victimas["Latitud"].round(____).astype(str) + ", " + victimas["Longitud"].round(3).astype(str)    # HUECO 13: cuántos decimales definen una celda de unos 110 metros

con_lugar = victimas["Latitud"].notna().values
pasado, futuro = victimas[(E | V) & con_lugar], victimas[P & con_lugar]
por_cruce = pasado.groupby("cruce").agg(siniestros=("mortal", "size"), mortales=("mortal", "sum"), esperados=("p", "sum"))
despues = futuro.groupby("cruce")["mortal"].sum()

desempate = np.random.default_rng(0).random(len(por_cruce))
def primeros(criterio):
    return por_cruce.assign(desempate=desempate).sort_values([criterio, "desempate"], ascending=False).index[:CONTROLES]

sorteo = np.random.default_rng(1)
replicas = {criterio: [] for criterio in ["siniestros", "mortales", "esperados"]}
escogidos = {criterio: set(primeros(criterio)) for criterio in replicas}
cruces_futuros = futuro["cruce"].values; mortal_futuro = futuro["mortal"].values
for _ in range(300):                                           # bootstrap sobre los siniestros de la prueba
    filas = sorteo.integers(0, len(futuro), len(futuro))
    for criterio in replicas:
        replicas[criterio].append(mortal_futuro[filas][np.isin(cruces_futuros[filas], list(escogidos[criterio]))].sum())

decision = pd.DataFrame({criterio: {"mortales de 2023 y 2024 en esos cruces": int(despues.reindex(list(escogidos[criterio])).fillna(0).sum()),
                                    "intervalo de 90 %": f"{np.percentile(replicas[criterio], 5):.0f} a {np.percentile(replicas[criterio], 95):.0f}",
                                    "% de todos los mortales": despues.reindex(list(escogidos[criterio])).fillna(0).sum() / mortal_futuro.sum() * 100} for criterio in replicas}).T
al_azar = np.mean([despues.reindex(por_cruce.sample(CONTROLES, random_state=s).index).fillna(0).sum() for s in range(200)])
print(f"Cruces con algún siniestro entre 2015 y 2022: {len(por_cruce)} | se escogen {CONTROLES} ({CONTROLES / len(por_cruce) * 100:.1f} %)")
print(f"Siniestros mortales de 2023 y 2024 con coordenadas: {int(mortal_futuro.sum())} | en {CONTROLES} cruces al azar caerían {al_azar:.1f}")
display(decision.round(1))
print("Cruces que comparten el criterio 1 y el 3:", len(escogidos["siniestros"] & escogidos["esperados"]))

Comparen con su tercera apuesta y respondan.

- ¿Cuál de los tres criterios atrapa más siniestros mortales futuros? ¿La diferencia entre ellos es mayor que sus intervalos? **Mi lectura:**
- Ordenar por los muertos del pasado parece lo más directo. ¿Por qué puede ser peor que ordenar por siniestros? Piensen en cuántos mortales hay por cruce. **Mi respuesta:**
- ¿Qué aporta el modelo frente a contar siniestros? Sean francos con lo que salió. **Mi respuesta:**

In [ ]:
recomendados = por_cruce.loc[list(primeros("esperados"))].head(10).round(2)
recomendados["dirección más frecuente"] = [pasado.loc[pasado["cruce"] == c, "Direccion"].mode().iloc[0] for c in recomendados.index]
display(recomendados)

### La decisión en tres frases

Para la secretaria de movilidad, que conoce la ciudad y no sabe qué es una curva de calibración.

1. **Recomendación:**
2. **Evidencia:**
3. **Riesgo:**

### 4.2 Bitácora de decisiones

Llenen las filas vacías. La primera va de ejemplo.

| Decisión | Alternativa descartada | Por qué |
|---|---|---|
| Solo siniestros con víctimas | Los tres niveles de gravedad | Los de solo daños dejaron de registrarse en 2023 |
| Gravedad a treinta días | | |
| El vacío en las columnas `Con_` es «no» | | |
| Hipótesis del agente fuera del modelo | | |
| Partición por años | | |
| Pérdida logarítmica para escoger | | |
| No balancear las clases | | |
| Coordenadas por defecto declaradas faltantes | | |
| Cruce: celda de tres decimales | | |
| Criterio para escoger los cruces | | |

### 4.3 Riesgos, ética y mantenimiento

- Un control en un cruce puede bajar los siniestros ahí y subirlos a dos cuadras. ¿Cómo lo sabrían?
- Hay siniestros que un indicador de gravedad marca con muertos y el otro no. ¿A quién le preguntarían cuál es el correcto, y qué resultado suyo cambiaría?
- Cerca del 4 % de los siniestros con víctimas no tiene coordenadas. ¿Qué pasa si faltan más en unas localidades que en otras?
- Los cruces con más siniestros registrados son también los de más tráfico y más presencia de policía. ¿Qué barrios pueden quedar por fuera de la lista por estar menos vigilados?
- Este modelo estima la probabilidad de muerte **dado** que hubo un siniestro con víctimas. ¿Por qué no sirve para decir en qué cruce es más probable que haya un siniestro?
- Detrás de cada fila mortal hay una familia. ¿Qué cuidados tendrían al publicar un mapa con estos puntos?

### 4.4 Cambien una línea

Vuelvan a la celda donde dice `ACTOR = "todos"`, pongan `ACTOR = "Con_Peaton"` y corran todo de nuevo con *Entorno de ejecución*, *Ejecutar todo*. Ahora la población son los siniestros en que hubo un peatón. Llenen la tabla con sus dos corridas.

| | Todos | Con peatón |
|---|---|---|
| Siniestros con víctimas y tasa de mortalidad | | |
| Ganador en validación | | |
| Pérdida logarítmica en prueba: ganador y tasa base | | |
| Mortales que predice el modelo balanceado, y mortales que hubo | | |
| Mortales futuros en los cruces escogidos, con cada criterio | | |
| % de todos los mortales que atrapan los cruces | | |

Con la tabla llena, respondan en tres líneas. ¿Los cien cruces para proteger peatones son los mismos que los cien cruces generales? ¿Qué le preguntarían a la Secretaría antes de decidir con cuál de las dos listas se queda? **Mi respuesta:**

Dejen el cuaderno entregado con `ACTOR = "Con_Peaton"` corrido y la tabla completa.

**Para entregar:** en Colab, menú *Archivo*, *Guardar una copia en GitHub*, y escojan su repositorio de portafolio, carpeta `clase04/`. Un cuaderno que no corre de principio a fin desde cero no se califica: antes de guardar usen *Entorno de ejecución*, *Reiniciar sesión y ejecutar todo*.